## 1. Document Preparation

In [1]:
print("Start RAG")

Start RAG


In [2]:
import os
os.environ['HF_HOME'] = 'G:/hf'

In [3]:
import pymupdf           #allows python to read pdfs


def pdf_to_markdown(pdf_path, markdown_path):
    
    
    markdown = []

    with pymupdf.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf, start=1):     #looping through every page giving each one a number

            #creating content for every page
            markdown.append(f"## Page {page_number}\n\n")
            markdown.append(page.get_text())
            markdown.append("\n\n")

    with open(markdown_path, "w", encoding="utf-8") as file:
        file.write("".join(markdown))

    print(f"Markdown file created: {markdown_path}")


pdf_to_markdown("C:/Users/PC/OneDrive/Desktop/egyptianhistory.pdf", "output.md")    #saving everything into output.md


Markdown file created: output.md


## 2. Chunking

In [4]:
from pathlib import Path
import re

INPUT_FILE = Path("output.md")
OUTPUT_FOLDER = Path("dataset")


BOOK_RANGES = [
    ("Book 1.Introduction", 43, 112),
    ("Book 2.The Old Kingdom", 113, 236),
    ("Book 3.The Middle Kingdom: The Feudal Age", 237, 312),
    ("Book 4.The Hyksos: The Rise of the Empire", 313, 338),
    ("Book 5.The Empire: First Period", 339, 548),
    ("Book 6.The Empire: Second Period", 549, 682),
    ("Book 7.The Decadence", 683, 748),
    ("Book 8.The Restoration and The End", 749, 784),
]


def get_book_name(page_number):     #taking page number and determines which book it belongs to
    """Helper Function"""
    """Return the book name for a given page number."""

    for book_name, first_page, last_page in BOOK_RANGES:
        if first_page <= page_number <= last_page:
            return book_name

    return None

def clean_text(text):
    """Helper Function"""
    """Clean the text by removing extra whitespace and unwanted characters."""

    text = re.sub(r'\s+', ' ', text)
    text = text.replace(r'[\r\n]+', '\n').strip()

    return text

 
def split_pages():    #each file contains book number, page number and its content
    """Split the Markdown file into separate files for each page."""

    text = INPUT_FILE.read_text(encoding="utf-8")

    pages = re.split(r"^##\s*Page\s+(\d+)\s*$", text, flags=re.MULTILINE)

    OUTPUT_FOLDER.mkdir(exist_ok=True)

    for i in range(1, len(pages), 2):

        page_number = int(pages[i])
        page_content = clean_text(pages[i + 1])
        book_name = get_book_name(page_number)

        if book_name:

            markdown = (
                f"# {book_name}\n\n"
                f"## Page {page_number}\n\n"
                f"{page_content}"
            )

            file_name = f"{book_name} - Page {page_number}.md"
            (OUTPUT_FOLDER / file_name).write_text(markdown, encoding="utf-8")

            

split_pages()


## 3. Embeddings

In [5]:
from sentence_transformers import SentenceTransformer
from pathlib import Path

DATASET_FOLDER = Path("dataset")
MODEL_NAME = "intfloat/multilingual-e5-small"          #embedding model


def read_page(file_path):     #reads each markdown file and extracts book name, page number and content
    lines = file_path.read_text(encoding="utf-8").splitlines()

    book_name = lines[0].replace("# ", "")
    page_number = int(lines[2].replace("## Page ", ""))
    content = " ".join(lines[3:]).strip()

    return {
        "book_name": book_name,
        "page_number": page_number,
        "content": content,
    }



files = sorted(DATASET_FOLDER.glob("*.md"))
pages = [read_page(file) for file in files]
texts = [f"passage: {page['content']}" for page in pages]

model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(          #coverting every page into a numerical vector
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
).tolist()



Batches:   0%|          | 0/10 [00:00<?, ?it/s]

In [6]:
from dotenv import load_dotenv
import os

load_dotenv()           #loading qdrant environment variables (URL, API COLLEC)

False

In [7]:
print("groq model:", os.getenv("GROQ_MODEL"))

groq model: None


In [11]:
#embeddings are stored in vector database


from qdrant_client import QdrantClient, models
from dotenv import load_dotenv
import os

load_dotenv()

load_dotenv(dotenv_path="G:/rag-project/project/.env")
QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
QDRANT_COLLECTION = os.getenv("QDRANT_COLLECTION")

print(repr(QDRANT_URL))


client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
)

collection_name = QDRANT_COLLECTION
vector_size = len(embeddings[0])

if not client.collection_exists(collection_name):
    client.create_collection(      #creating my embeddings collection egyptian-history-rag
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size=vector_size,
            distance=models.Distance.COSINE,
        ),
    )


#diff between point and payload 

points = [
    models.PointStruct(
        id=index,
        vector=embedding,
        payload=page,
    )
    for index, (page, embedding) in enumerate(zip(pages, embeddings))
]

batch_size = 100

for start in range(0, len(points), batch_size):
    batch = points[start:start + batch_size]
    client.upsert(    #uploading the pages and embeddings into qdrant
        collection_name=collection_name,
        points=batch,
    )

print(f"Uploaded {len(points)} pages to Qdrant.")

'https://0c8a89ea-6e43-40ce-9485-4f75695e984d.europe-west6-0.gcp.cloud.qdrant.io'
Uploaded 296 pages to Qdrant.


## 4. Query Router

* ### Deciding what to do with user's question(query)

In [26]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

load_dotenv()

query = input("Ask a question: " )

router_llm = ChatGroq(          #llm here (groq) receives the q and classifies it
    model=os.getenv("GROQ_MODEL"),
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0,
)

SYSTEM_PROMPT = """You are a query router for an Egyptian History RAG system.
Your task is to classify the user's query into exactly one of these three categories:

1. "retrieve" — Use this when the user is asking a question related to Egyptian history that should be answered using information from the provided Egyptian History documents. This includes questions about pharaohs, dynasties, kingdoms, wars, rulers, historical events, places, civilizations, religion, culture, monuments, or other historical information covered by the documents.

2. "chitchat" — Use this for casual conversation that does not require information from the Egyptian History documents, such as greetings, thanks, or general conversational messages.

3. "off-topic" — Use this when the question is unrelated to Egyptian history and cannot be answered using the Egyptian History documents.

Return ONLY one of these exact labels:
retrieve
chitchat
off-topic

Examples:

User: "Who was Ramses II?"
Output: retrieve

User: "What happened during the Old Kingdom?"
Output: retrieve

User: "Tell me about the pyramids."
Output: retrieve

User: "Hello!"
Output: chitchat

User: "Thank you."
Output: chitchat

User: "What is the capital of France?"
Output: off-topic

User: "How do neural networks work?"
Output: off-topic"""


router_messages = [
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content=query),
]


route = router_llm.invoke(router_messages).content.strip().lower()
route = route.splitlines()[0].strip(" `.,:")

if route not in {"retrieve", "chitchat", "off-topic"}:
    route = "off-topic"

print("Route:", route)


Route: retrieve


## 4.1. Retrieval stage

* #### Running if and only if the route was retrieve (user asked a question related to topic)

In [27]:
import os
from dotenv import load_dotenv


load_dotenv()
top_k = 3

if route == "retrieve":
    query_vector = model.encode(
        [f"query: {query}"],
        normalize_embeddings=True,
    )[0].tolist()

    results = client.query_points(         #searches qdrant for most similar vectors
        collection_name=QDRANT_COLLECTION,
        query=query_vector,
        limit=top_k,
    ).points


    context = ""
    for result in results:
        page = result.payload
        context += (
            f"Book: {page['book_name']}\n"
            f"Page: {page['page_number']}\n"
            f"Content: {page['content']}\n\n"
        )

        print("Score:", result.score)
        print("Book:", page["book_name"])
        print("Page:", page["page_number"])
        print("Content:", page["content"])
        print("-" * 80)
        
else:
    print("No database search needed.")


Score: 0.86165893
Book: Book 2.The Old Kingdom
Page: 199
Content: THE PYRAMID BUILDERS anes, supplanting the powerful Snefru and becoming the founder of a new line. We only see him looming grandly from the obscure array of Pharaohs of his time, his greatness pro- claimed by the noble tomb which he erected at Gizeh, oppo- site modern Cairo. It has now become the chief project of the state to furnish a vast, impenetrable and indestructible resting place for the body of the king, who concentrated upon this enterprise the greatest resources of wealth, skill and labour at his command. How strong and effective must have been the organization of Khufu’s government we appreciate in some measure when we learn that his pyramid contains some two million three hundred thousand blocks, each weigh- ing on the average two and a half tons.*’ The mere organiza- tion of labour involved in the quarrying, transportation and proper assembly of this vast mass of material is a task which in itself must have 

## 5. Keyword Search

In [28]:
if route == "retrieve":
    keyword_query = "Great Pyramid"
    top_k = 3

    keywords = keyword_query.lower().split()
    keyword_results = []

    for page in pages:
        content = page["content"].lower()
        score = sum(content.count(keyword) for keyword in keywords)

        if score > 0:
            keyword_results.append({
                "score": score,
                "book_name": page["book_name"],
                "page_number": page["page_number"],
                "content": page["content"],
            })

    keyword_results.sort(key=lambda result: result["score"], reverse=True)   #highest scoring page first

    for result in keyword_results[:top_k]:
        print("Keyword score:", result["score"])
        print("Book:", result["book_name"])
        print("Page:", result["page_number"])
        print("Content:", result["content"])
        print("-" * 80)


Keyword score: 9
Book: Book 2.The Old Kingdom
Page: 199
Content: THE PYRAMID BUILDERS anes, supplanting the powerful Snefru and becoming the founder of a new line. We only see him looming grandly from the obscure array of Pharaohs of his time, his greatness pro- claimed by the noble tomb which he erected at Gizeh, oppo- site modern Cairo. It has now become the chief project of the state to furnish a vast, impenetrable and indestructible resting place for the body of the king, who concentrated upon this enterprise the greatest resources of wealth, skill and labour at his command. How strong and effective must have been the organization of Khufu’s government we appreciate in some measure when we learn that his pyramid contains some two million three hundred thousand blocks, each weigh- ing on the average two and a half tons.*’ The mere organiza- tion of labour involved in the quarrying, transportation and proper assembly of this vast mass of material is a task which in itself must have s

## 6. Generation

- ### where RAG system actually answers teh user's question fromn the retrieved pages

In [16]:
if route == "retrieve":
    from langchain_google_genai import ChatGoogleGenerativeAI
    from langchain_core.messages import SystemMessage, HumanMessage

    gemini_llm = ChatGoogleGenerativeAI(
        model=os.getenv("GEMINI_MODEL"),
        api_key=os.getenv("GEMINI_API_KEY"),
        temperature=0,
    )

    messages = [
        SystemMessage(
            content="Answer only from the provided pages. If the answer is not there, say you do not know. Keep the answer concise."
        ),
        HumanMessage(
            content=f"Context:\n{context}\nQuestion:\n{query}"
        ),
    ]

    response = gemini_llm.invoke(messages)
    print("\nAnswer:")
    print(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.



Answer:
Khufu built the great pyramid.


## 7. Retrieval Evaluation: Precision and Recall

In [20]:
evaluation_cases = [
    {
        "query": "Who are the hyksos?",
        "relevant_pages": {614},
    },
    {
        "query": "Where is the Keftyew?",
        "relevant_pages": {101, 112},
    },
    {
        "query": "Who won in the war of Ramses II?",
        "relevant_pages": {607, 608, 609},
    },
]

top_k = 3
precision_scores = []
recall_scores = []
retrieved_for_evaluation = []

for case in evaluation_cases:
    query_vector = model.encode(
        [f"query: {case['query']}"],
        normalize_embeddings=True,
    )[0].tolist()

    search_results = client.query_points(
        collection_name=QDRANT_COLLECTION,
        query=query_vector,
        limit=top_k,
        with_payload=True,
    ).points

    retrieved_pages = {result.payload["page_number"] for result in search_results}
    relevant_pages = case["relevant_pages"]
    relevant_retrieved = retrieved_pages & relevant_pages

    precision = len(relevant_retrieved) / len(retrieved_pages) if retrieved_pages else 0
    recall = len(relevant_retrieved) / len(relevant_pages)

    precision_scores.append(precision)
    recall_scores.append(recall)
    retrieved_for_evaluation.append((case, search_results))

    print(case["query"])
    print("Expected pages:", relevant_pages)
    print("Retrieved pages:", retrieved_pages)
    print(f"Precision: {precision:.2f}")
    print(f"Recall:    {recall:.2f}")
    print("-" * 60)

print(f"Average precision: {sum(precision_scores) / len(precision_scores):.2f}")
print(f"Average recall:    {sum(recall_scores) / len(recall_scores):.2f}")


Who are the hyksos?
Expected pages: {614}
Retrieved pages: {718, 614, 63}
Precision: 0.33
Recall:    1.00
------------------------------------------------------------
Where is the Keftyew?
Expected pages: {112, 101}
Retrieved pages: {112, 60, 54}
Precision: 0.33
Recall:    0.50
------------------------------------------------------------
Who won in the war of Ramses II?
Expected pages: {608, 609, 607}
Retrieved pages: {609, 607, 615}
Precision: 0.67
Recall:    0.67
------------------------------------------------------------
Average precision: 0.44
Average recall:    0.72


## 8. LLM-as-a-Judge Evaluation

### Final evaluation stage (Gemini acting as a judge)
- ##### it receives context + question + generated answer and evaluates the answer 

In [21]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage

gemini_llm = ChatGoogleGenerativeAI(
    model=os.getenv("GEMINI_MODEL"),
    api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0,
)

for case, search_results in retrieved_for_evaluation:
    context = "\n\n".join(
        f"Page {result.payload['page_number']}: {result.payload['content']}"
        for result in search_results
    )

    answer = gemini_llm.invoke([
        SystemMessage(content="Answer only from the provided context. If the answer is not there, say you do not know."),
        HumanMessage(content=f"Context:\n{context}\n\nQuestion:\n{case['query']}"),
    ]).text

    judge = gemini_llm.invoke([
        SystemMessage(content="""You are an evaluator for a question-answering system.
                                Judge the answer using only the context.
                                Return exactly this format:
                                Score: X/5
                                Grounded: yes or no
                                Reason: one short sentence"""),
        HumanMessage(content=f"Context:\n{context}\n\nQuestion:\n{case['query']}\n\nAnswer:\n{answer}"),
    ]).text

    print("Question:", case["query"])
    print("Answer:", answer)
    print("Judge:", judge)
    print("-" * 60)


Question: Who are the hyksos?
Answer: The Hyksos were a line of rulers from Asia who entered and appropriated Egypt, maintaining themselves for perhaps a century. They resided at Avaris in the eastern Delta and were responsible for the importation of the horse into Egypt.
Judge: Score: 5/5
Grounded: yes
Reason: The answer accurately summarizes the information provided in the text regarding the origin, duration, residence, and impact of the Hyksos.
------------------------------------------------------------
Question: Where is the Keftyew?
Answer: I do not know.
Judge: Score: 5/5
Grounded: yes
Reason: The provided context pages are empty, so the correct answer is that the information is not available.
------------------------------------------------------------
Question: Who won in the war of Ramses II?
Answer: I do not know.
Judge: Score: 5/5
Grounded: yes
Reason: The provided text does not contain information regarding the outcome of any war involving Ramses II.
----------------------